In [199]:
import tensorflow as tf
import numpy as np
import pandas as pd

In [200]:
df = pd.read_csv('/content/ai4i2020.csv')

In [201]:
df.keys()

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='object')

In [202]:
df.nunique()

,0
UDI,10000
Product ID,10000
Type,3
Air temperature [K],93
Process temperature [K],82
Rotational speed [rpm],941
Torque [Nm],577
Tool wear [min],246
Machine failure,2
TWF,2


In [203]:
df.drop(columns = ['UDI','Product ID','HDF','OSF','RNF','TWF','Tool wear [min]','PWF'],inplace = True)

In [204]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Type                     10000 non-null  object 
 1   Air temperature [K]      10000 non-null  float64
 2   Process temperature [K]  10000 non-null  float64
 3   Rotational speed [rpm]   10000 non-null  int64  
 4   Torque [Nm]              10000 non-null  float64
 5   Machine failure          10000 non-null  int64  
dtypes: float64(3), int64(2), object(1)
memory usage: 468.9+ KB


In [205]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [206]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer([('encoder', OneHotEncoder(), [0])], remainder='passthrough')
x = ct.fit_transform(x)

In [207]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 0)

In [208]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [209]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)

In [210]:
ann = tf.keras.models.Sequential()

In [211]:
ann.add(tf.keras.layers.Dense(units = 64, activation = 'relu'))
ann.add(tf.keras.layers.Dense(units = 32, activation = 'relu'))
ann.add(tf.keras.layers.Dense(units = 16, activation = 'relu'))
ann.add(tf.keras.layers.Dense(units = 8, activation = 'relu'))
ann.add(tf.keras.layers.Dense(units = 1, activation = 'sigmoid'))

In [212]:
ann.compile(loss = 'binary_crossentropy',optimizer = 'adam',metrics = ['accuracy'])


In [213]:
ann.fit(x_train_balanced,y_train_balanced,batch_size = 16, epochs = 100)

Epoch 1/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8426 - loss: 0.3707
Epoch 2/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8793 - loss: 0.3080
Epoch 3/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8818 - loss: 0.2895
Epoch 4/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8853 - loss: 0.2726
Epoch 5/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8869 - loss: 0.2580
Epoch 6/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8923 - loss: 0.2431
Epoch 7/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8969 - loss: 0.2324
Epoch 8/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9006 - loss: 0.2258
Epoch 9/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9029 - loss: 0.2163
Epoch 10/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9077 - loss: 0.2097
Epoch 11/100
967/967 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9103 - loss: 0.2047
Epoch 12/100
967/967 ━━━━━━━━━━━━━━━━━━━━

In [178]:
predictn = ann.predict(x_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [179]:
predictn = (predictn > 0.5)

In [180]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
print(confusion_matrix(y_test,predictn))
print(accuracy_score(y_test,predictn))

[[1794  131]
 [  31   44]]
0.919


In [181]:
print(classification_report(y_test,predictn))

              precision    recall  f1-score   support

           0       0.98      0.93      0.96      1925
           1       0.25      0.59      0.35        75

    accuracy                           0.92      2000
   macro avg       0.62      0.76      0.65      2000
weighted avg       0.96      0.92      0.93      2000



In [218]:
import joblib as jb

ann.save("Neural_Network.h5")

In [216]:
jb.dump(sc,"Feature_Scaling.pkl")

['Feature_Scaling.pkl']

In [214]:
jb.dump(ct,"OneHotEncoder_new.pkl")

['OneHotEncoder_new.pkl']

In [197]:
print(type(ct))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [217]:
sample = pd.DataFrame({
    "Type":["L"],
    "Air temperature [K]":[298.2],
    "Process temperature [K]":[308.7],
    "Rotational speed [rpm]":[1500],
    "Torque [Nm]":[40]
})

sample = ct.transform(sample)
sample = sc.transform(sample)

print(ann.predict(sample))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step
[[0.00030225]]
